# Training Loop Anatomy

The core cycle: **forward → loss → backward → optimizer.step()**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class DummyDataGenerator:
    """Synthetic classification data for CPU-only tutorials."""
    def __init__(self, n_samples=256, n_features=8, n_classes=3, seed=42):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n_samples, n_features, generator=g)
        self.y = torch.randint(0, n_classes, (n_samples,), generator=g)

    def tensors(self):
        return self.X, self.y

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class SimpleMLP(nn.Module):
    def __init__(self, in_dim=8, hidden=16, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)


## 1. Setup model, loss, optimizer, data

In [ ]:
gen = DummyDataGenerator(n_samples=200, n_features=8, n_classes=3)
ds = TabularDataset(*gen.tensors())
loader = DataLoader(ds, batch_size=32, shuffle=True)
model = SimpleMLP(in_dim=8, hidden=16, n_classes=3)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
loss_history = []

## 2. One epoch — step by step

In [ ]:
for epoch in range(1):
    for batch_x, batch_y in loader:
        # 1. Forward
        logits = model(batch_x)
        # 2. Loss
        loss = criterion(logits, batch_y)
        # 3. Backward
        opt.zero_grad()
        loss.backward()
        # 4. Update
        opt.step()
        loss_history.append(loss.item())
print(f"Steps: {len(loss_history)}, final loss: {loss_history[-1]:.4f}")

## 3. Loss per training step

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(loss_history, color='steelblue', lw=1.5)
ax.set_xlabel('step'); ax.set_ylabel('loss')
ax.set_title('Loss during one epoch')
plt.tight_layout(); plt.show()

## 4. Annotated pipeline diagram (conceptual)

In [ ]:
steps = ['forward', 'loss', 'backward', 'step']
colors = ['#3498db', '#e74c3c', '#9b59b6', '#2ecc71']
fig, ax = plt.subplots(figsize=(10, 2.5))
for i, (s, c) in enumerate(zip(steps, colors)):
    ax.add_patch(plt.Rectangle((i*2.2, 0.2), 1.8, 0.6, fc=c, ec='black', alpha=0.8))
    ax.text(i*2.2 + 0.9, 0.5, s, ha='center', va='center', color='white', fontsize=12, fontweight='bold')
    if i < 3:
        ax.annotate('', xy=((i+1)*2.2, 0.5), xytext=(i*2.2+1.8, 0.5),
                    arrowprops=dict(arrowstyle='->', lw=2))
ax.set_xlim(-0.2, 9); ax.set_ylim(0, 1); ax.axis('off')
ax.set_title('Training step pipeline')
plt.tight_layout(); plt.show()

## 5. Multi-epoch training curve

In [ ]:
epoch_losses = []
for epoch in range(5):
    ep_loss = []
    for bx, by in loader:
        opt.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward(); opt.step()
        ep_loss.append(loss.item())
    epoch_losses.append(np.mean(ep_loss))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 6), epoch_losses, 'o-', color='coral', lw=2)
ax.set_xlabel('epoch'); ax.set_ylabel('mean loss')
ax.set_title('Loss decreases over epochs')
plt.tight_layout(); plt.show()